<a href="https://colab.research.google.com/github/leadministrador/mineria-datos-ii-proyecto-integrador/blob/main/notebooks/02_streaming_bronze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip -q install pyspark

In [5]:
!unzip -q -o cloud_provider_challenge_dataset_v1.zip -d /content/proyecto

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("proyecto-integrador")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Versión de Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Versión de Spark: 4.0.4
Master: local[*]
Shuffle partitions: 8


In [7]:
LANDING = "/content/proyecto/datalake/landing"
eventos = spark.read.json(LANDING + "/usage_events_stream")
eventos.printSchema()
print("Eventos:", eventos.count())

root
 |-- carbon_kg: double (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- event_id: string (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- metric: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- schema_version: long (nullable = true)
 |-- service: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: string (nullable = true)

Eventos: 43200


In [9]:
esquema = ("event_id STRING, timestamp STRING, org_id STRING, resource_id STRING, "
           "service STRING, region STRING, metric STRING, value STRING, unit STRING, "
           "cost_usd_increment DECIMAL(10,4), schema_version BIGINT, "
           "carbon_kg DECIMAL(10,4), genai_tokens BIGINT")
print("Esquema listo")

Esquema listo


In [10]:
BRONZE = "/content/proyecto/datalake/bronze/usage_events"
CHECK = "/content/proyecto/checkpoints/bronze_usage_events"

def ingesta():
    eventos_stream = (spark.readStream
        .schema(esquema)
        .option("maxFilesPerTrigger", 10)
        .json(LANDING + "/usage_events_stream")
        .withColumn("event_ts", F.col("timestamp").cast("timestamp"))
        .withColumn("event_date", F.to_date("event_ts"))
        .withColumn("ingest_ts", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_name"))
        .withWatermark("event_ts", "60 days")
        .dropDuplicatesWithinWatermark(["event_id"]))
    q = (eventos_stream.writeStream
        .format("parquet")
        .option("path", BRONZE)
        .option("checkpointLocation", CHECK)
        .partitionBy("event_date")
        .outputMode("append")
        .trigger(availableNow=True)
        .start())
    q.awaitTermination()

ingesta()
print("Filas en Bronze:", spark.read.parquet(BRONZE).count())

Filas en Bronze: 43200


In [11]:
antes = spark.read.parquet(BRONZE).count()
ingesta()
despues = spark.read.parquet(BRONZE).count()
print("Antes:", antes, "· Después:", despues)

spark.read.parquet(BRONZE).select("event_id", "event_date", "source_file", "ingest_ts").show(3, truncate=False)

Antes: 43200 · Después: 43200
+----------------+----------+----------------------+-----------------------+
|event_id        |event_date|source_file           |ingest_ts              |
+----------------+----------+----------------------+-----------------------+
|evt_ecrixjiey5mp|2025-08-02|events_part_0045.jsonl|2026-10-30 02:07:17.503|
|evt_alhdhfrsbyst|2025-08-02|events_part_0045.jsonl|2026-10-30 02:07:17.503|
|evt_djk15fyiicmp|2025-08-02|events_part_0036.jsonl|2026-10-30 02:07:17.503|
+----------------+----------+----------------------+-----------------------+
only showing top 3 rows


In [12]:
SILVER = "/content/proyecto/datalake/silver/usage_events"
QUARANTINE = "/content/proyecto/datalake/quarantine/usage_events"

esquema_bronze = esquema + ", event_ts TIMESTAMP, event_date DATE, ingest_ts TIMESTAMP, source_file STRING"

RULE = ("coalesce(event_id IS NOT NULL AND cost_usd_increment >= -0.01 "
        "AND (value IS NULL OR unit IS NOT NULL), false)")

orgs = (spark.read.option("header", True).option("inferSchema", True)
        .csv(LANDING + "/customers_orgs.csv")
        .select("org_id", "industry", "plan_tier"))

bronze_stream = spark.readStream.schema(esquema_bronze).parquet(BRONZE)

silver = (bronze_stream.filter(RULE)
          .withColumn("value", F.col("value").cast("decimal(12,4)"))
          .join(F.broadcast(orgs), "org_id", "left"))
quarantine = bronze_stream.filter(f"NOT {RULE}")

(silver.writeStream.format("parquet")
    .option("path", SILVER)
    .option("checkpointLocation", "/content/proyecto/checkpoints/silver_usage_events")
    .partitionBy("event_date")
    .trigger(availableNow=True).start().awaitTermination())

(quarantine.writeStream.format("parquet")
    .option("path", QUARANTINE)
    .option("checkpointLocation", "/content/proyecto/checkpoints/quarantine_usage_events")
    .trigger(availableNow=True).start().awaitTermination())

print("Silver:", spark.read.parquet(SILVER).count())
print("Quarantine:", spark.read.parquet(QUARANTINE).count())

Silver: 40956
Quarantine: 2244


In [13]:
for tabla in ["customers_orgs", "users", "billing_monthly"]:
    df = (spark.read.option("header", True).option("inferSchema", True)
          .csv(LANDING + "/" + tabla + ".csv")
          .withColumn("ingest_ts", F.current_timestamp())
          .withColumn("source_file", F.col("_metadata.file_name")))
    df.write.mode("overwrite").parquet("/content/proyecto/datalake/bronze/" + tabla)
    print(tabla, spark.read.parquet("/content/proyecto/datalake/bronze/" + tabla).count())

customers_orgs 80
users 800
billing_monthly 240


In [14]:
def leer(tabla, esquema_m, clave):
    return (spark.read.option("header", True).schema(esquema_m)
            .csv(LANDING + "/" + tabla + ".csv")
            .dropDuplicates([clave])
            .withColumn("ingest_ts", F.current_timestamp())
            .withColumn("source_file", F.col("_metadata.file_name")))

BRONZE_M = "/content/proyecto/datalake/bronze/"

In [15]:
leer("customers_orgs",
     "org_id STRING, org_name STRING, industry STRING, hq_region STRING, "
     "plan_tier STRING, is_enterprise STRING, signup_date DATE, sales_rep STRING, "
     "lifecycle_stage STRING, marketing_source STRING, nps_score DECIMAL(5,1)",
     "org_id").write.mode("overwrite").parquet(BRONZE_M + "customers_orgs")

print("customers_orgs", spark.read.parquet(BRONZE_M + "customers_orgs").count())

customers_orgs 80


In [16]:
leer("users",
     "user_id STRING, org_id STRING, email STRING, role STRING, active STRING, "
     "created_at DATE, last_login DATE",
     "user_id").write.mode("overwrite").parquet(BRONZE_M + "users")

print("users", spark.read.parquet(BRONZE_M + "users").count())

users 800


In [17]:
leer("billing_monthly",
     "invoice_id STRING, org_id STRING, month DATE, subtotal DECIMAL(12,2), "
     "credits DECIMAL(12,2), taxes DECIMAL(12,2), currency STRING, "
     "exchange_rate_to_usd DECIMAL(10,5)",
     "invoice_id").write.mode("overwrite").partitionBy("month").parquet(BRONZE_M + "billing_monthly")

print("billing_monthly", spark.read.parquet(BRONZE_M + "billing_monthly").count())

billing_monthly 240
